# Automatic Vertebrae Fracture Screening - Colab runner
Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom. Nothing here has been run on real VerSe data yet.

In [ ]:
!git clone https://github.com/weasley29/vertfrac.git
%cd vertfrac
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
!pytest -q tests      # everything should pass before you touch real data

## 1. Data (VerSe). ~30-40 GB: mount Google Drive or use a big disk.

In [ ]:
!bash scripts/01_download_verse.sh
!python scripts/02_inspect_dataset.py

### Genant grades
The zips have NO grades. Download the fracture-grade metadata from the VerSe'19 OSF project https://osf.io/nqjyw/ (Loffler et al. 2020) into `data/`, then set the file name below.

In [ ]:
GRADES_FILE = 'data/PUT_FILE_NAME_HERE'
!python scripts/02b_build_grades_csv.py --file {GRADES_FILE}
# check: paper says VerSe'19 = 160 scans / 141 patients / 1725 vertebrae

In [ ]:
!python scripts/03_extract_rois.py --limit 5   # quick check
!python scripts/03_extract_rois.py

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
m = pd.read_csv('data/manifest.csv'); m = m[m.grade>=0]
fig, ax = plt.subplots(4, 8, figsize=(16, 8))
for g in range(4):
    s = m[m.grade==g].sample(min(8, (m.grade==g).sum()), random_state=0)
    for j, p in enumerate(s.roi_path): ax[g, j].imshow(np.load(p)[1], cmap='gray'); ax[g, j].axis('off')
    ax[g, 0].set_title(f'grade {g}', loc='left')
plt.show()   # vertebrae centred and upright? bone visible?

## 2. Train: baseline vs SupCon (HF microsoft/resnet-50)

In [ ]:
!python scripts/04_train_classifier.py --lam 0.0 --out runs/baseline --amp
!python scripts/04_train_classifier.py --lam 0.5 --out runs/supcon --amp

In [ ]:
!python scripts/05_gradcam_demo.py --run runs/supcon --n 8
from IPython.display import Image; Image('gradcam_demo.png')

## 3. Ablation + tables (long: trim the grid in scripts/07_run_ablation.py if needed)

In [ ]:
!python scripts/07_run_ablation.py --extra --amp
!python scripts/08_make_tables.py

## 4. Segmentation and end-to-end

In [ ]:
!python scripts/06_train_unet.py

In [ ]:
!python scripts/10_eval_pred_masks.py --run runs/supcon --seg-ckpt runs/unet/best.pt --split test

In [ ]:
import glob
ct = sorted(glob.glob('data/verse/**/*_ct.nii.gz', recursive=True))[0]
!python scripts/11_screen_volume.py --ct {ct} --run runs/supcon --seg-ckpt runs/unet/best.pt --out screening
from IPython.display import Image; Image('screening/overlay.png')